# Vibe Quest S13 · ¿Qué pasa con un Transformer sin Positional Encoding?
**Pregunta 2 (Clase 12):** si quitamos el Positional Encoding, "El gato persiguió al perro" y "El perro persiguió al gato" ¿producen lo mismo? Lo demuestro con código en PyTorch, comparo los embeddings de salida con y sin PE y visualizo por qué se pierde el orden.

**Hipótesis (la que la Clase 12 plantea en la lámina 120):** la self-attention no distingue el orden; sin PE, permutar las palabras solo permuta las filas de la salida. Con PE, el orden queda codificado y las salidas cambian.

Todo corre en CPU. Los pesos son aleatorios con semilla fija (no se entrena nada), así que los resultados son reproducibles.

# 1 · Librerías
`torch` para el modelo, `numpy` para generar los pesos con semilla fija (así coinciden con la referencia en NumPy), `pandas` para tablas, `matplotlib` para los gráficos e `ipywidgets` para los sliders. Todo viene instalado en Google Colab.

In [ ]:
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

torch.set_default_dtype(torch.float64)
print("torch", torch.__version__)

# 2 · Parámetros
- **`D`:** dimensión del embedding (`d_model`). **`CAPAS`:** capas del codificador. **`SEMILLA`:** semilla de los pesos.
- **`FRASE_A` y `FRASE_B`:** las dos frases de la consigna. Tienen las mismas palabras y distinto orden.
- **`FRASES_EXTRA`:** una frase más larga y una con una palabra repetida, para probar que el resultado no depende de las frases de la consigna.

In [ ]:
D = 16
CAPAS = 2
SEMILLA = 0
FRASE_A = ["El", "gato", "persiguió", "al", "perro"]
FRASE_B = ["El", "perro", "persiguió", "al", "gato"]
FRASES_EXTRA = {
    "larga_7": ["El", "gato", "negro", "persiguió", "al", "perro", "grande"],
    "repetida": ["El", "gato", "vio", "al", "gato", "gris"],
}
CARPETA_SALIDA = Path("resultados_s13")
CARPETA_SALIDA.mkdir(exist_ok=True)

# 3 · Positional Encoding sinusoidal
Es la fórmula de la Clase 12 (lámina 120): `PE(pos, 2i) = sin(pos / 10000^(2i/d))` y `PE(pos, 2i+1) = cos(pos / 10000^(2i/d))`. Cada posición recibe un vector distinto que se suma al embedding de la palabra.

In [ ]:
def pe_sinusoidal(n_pos, d):
    pos = torch.arange(n_pos)[:, None].double()
    i = torch.arange(0, d, 2)[None, :].double()
    ang = pos / (10000.0 ** (i / d))
    pe = torch.zeros(n_pos, d)
    pe[:, 0::2] = torch.sin(ang)
    pe[:, 1::2] = torch.cos(ang)
    return pe

## 3 · b · Comprobación contra la fórmula
Calculo a mano algunos valores de la fórmula y verifico que la función los reproduce: en `pos = 0` los senos valen 0 y los cosenos 1, y en `pos = 3`, `i = 1` (d = 8) el seno es `sin(3 / 10000^(2/8))`.

In [ ]:
pe_prueba = pe_sinusoidal(10, 8)
assert torch.allclose(pe_prueba[0, 0::2], torch.zeros(4))
assert torch.allclose(pe_prueba[0, 1::2], torch.ones(4))
esperado = np.sin(3 / 10000 ** (2 / 8))
assert abs(pe_prueba[3, 2].item() - esperado) < 1e-12
print("PE(3, 2) =", pe_prueba[3, 2].item(), "| fórmula =", esperado)
print("PE verificado contra la fórmula de la lámina 120")

# 4 · El modelo: un codificador Transformer mínimo
Cada capa hace: self-attention `softmax(QKᵀ/√d)V`, luego `LayerNorm(x + Sublayer(x))` (Add & Norm de la lámina 122) y una red feed-forward con su Add & Norm. Los embeddings de las palabras y los pesos son aleatorios con semilla fija, generados con NumPy para que coincidan con la referencia. Un mismo token tiene **siempre el mismo embedding**, esté donde esté en la frase.

In [ ]:
def crear_pesos(d, n_capas, semilla=0):
    rng = np.random.default_rng(semilla)
    d_ff = 4 * d
    capas = []
    for _ in range(n_capas):
        capas.append({
            "Wq": rng.normal(0, d ** -0.5, (d, d)), "Wk": rng.normal(0, d ** -0.5, (d, d)),
            "Wv": rng.normal(0, d ** -0.5, (d, d)), "Wo": rng.normal(0, d ** -0.5, (d, d)),
            "W1": rng.normal(0, d ** -0.5, (d, d_ff)), "W2": rng.normal(0, d_ff ** -0.5, (d_ff, d)),
        })
    return [{k: torch.tensor(v) for k, v in capa.items()} for capa in capas]


def crear_embeddings(vocab, d, semilla=1, escala=1.0):
    rng = np.random.default_rng(semilla)
    return {w: torch.tensor(escala * rng.normal(0, 1, d)) for w in sorted(set(vocab))}


def codificador(x, capas):
    d = x.shape[1]
    atenciones = []
    for p in capas:
        q, k, v = x @ p["Wq"], x @ p["Wk"], x @ p["Wv"]
        a = torch.softmax(q @ k.T / d ** 0.5, dim=-1)
        atenciones.append(a)
        x = F.layer_norm(x + (a @ v) @ p["Wo"], (d,))
        x = F.layer_norm(x + torch.relu(x @ p["W1"]) @ p["W2"], (d,))
    return x, atenciones


def ejecutar(frase, emb, capas, usar_pe, escala_pe=1.0):
    x = torch.stack([emb[w] for w in frase])
    if usar_pe:
        x = x + escala_pe * pe_sinusoidal(len(frase), x.shape[1])
    return codificador(x, capas)

## 4 · b · Medidas que uso
- **Permutación `perm`:** índice de cada palabra de la frase B dentro de la frase A (para B = `[El, perro, persiguió, al, gato]` queda `[0, 4, 2, 3, 1]`).
- **Error de equivarianza:** `max |salida(B) − salida(A)[perm]|`. Si es ~0, la salida de cada palabra no depende de su posición: el modelo solo reordenó las filas.
- **Diferencia de la frase:** `max |media de filas de A − media de filas de B|`. Si es ~0, las dos frases se ven idénticas a un clasificador que promedie los vectores.
- **Coseno de "gato":** similitud entre el vector de salida de "gato" en A (posición 1) y en B (posición 4).

In [ ]:
def permutacion(frase_a, frase_b):
    return [frase_a.index(w) for w in frase_b]


def coseno(a, b):
    return (a @ b / (a.norm() * b.norm())).item()


def medir(frase_a, frase_b, emb, capas, usar_pe, escala_pe=1.0):
    perm = permutacion(frase_a, frase_b)
    sa, _ = ejecutar(frase_a, emb, capas, usar_pe, escala_pe)
    sb, _ = ejecutar(frase_b, emb, capas, usar_pe, escala_pe)
    return {
        "error_equivarianza": (sb - sa[perm]).abs().max().item(),
        "dif_promedio_frase": (sa.mean(0) - sb.mean(0)).abs().max().item(),
    }

# 5 · Experimento 1 · Las dos frases de la consigna
Ejecuto el mismo modelo, con los mismos pesos, sobre A y B, sin PE y con PE. Si la hipótesis es correcta, sin PE el error de equivarianza y la diferencia de la frase deben ser de orden 1e-15 (ruido de punto flotante).

In [ ]:
emb = crear_embeddings(FRASE_A, D)
capas = crear_pesos(D, CAPAS, SEMILLA)
PERM = permutacion(FRASE_A, FRASE_B)
print("perm A → B:", PERM)

filas = []
for usar_pe in (False, True):
    fila = {"usar_pe": usar_pe, **medir(FRASE_A, FRASE_B, emb, capas, usar_pe)}
    sa, _ = ejecutar(FRASE_A, emb, capas, usar_pe)
    sb, _ = ejecutar(FRASE_B, emb, capas, usar_pe)
    fila["cos_gato_A_vs_B"] = coseno(sa[1], sb[4])
    filas.append(fila)
e1 = pd.DataFrame(filas)
display(e1)

## 5 · b · Verificación con `assert`
Sin PE: la salida de B es exactamente la de A con las filas reordenadas y el "gato" tiene el mismo vector en las dos frases. Con PE: las salidas difieren de forma clara. Si algo de esto fallara, la celda se detiene.

In [ ]:
sin_pe, con_pe = e1.iloc[0], e1.iloc[1]
assert sin_pe.error_equivarianza < 1e-10 and sin_pe.dif_promedio_frase < 1e-10
assert abs(sin_pe.cos_gato_A_vs_B - 1.0) < 1e-10
assert con_pe.error_equivarianza > 0.1 and con_pe.dif_promedio_frase > 0.01
print("Sin PE el orden no deja huella; con PE sí")

# 6 · Visualización 1 · Matrices de atención de la capa 1
Cada fila es una palabra (la consulta) y cada columna, la palabra a la que atiende. Sin PE, la matriz de B es la de A con filas y columnas reordenadas: nada cambia salvo el orden de las etiquetas. Con PE, la matriz de B ya es otra.

In [ ]:
fig, ejes = plt.subplots(2, 2, figsize=(9, 8))
for r, (frase, nombre) in enumerate([(FRASE_A, "A"), (FRASE_B, "B")]):
    for c, usar_pe in enumerate((False, True)):
        _, att = ejecutar(frase, emb, capas, usar_pe)
        ejes[r, c].imshow(att[0].numpy(), cmap="viridis")
        ejes[r, c].set_xticks(range(len(frase)))
        ejes[r, c].set_yticks(range(len(frase)))
        ejes[r, c].set_xticklabels(frase, rotation=45)
        ejes[r, c].set_yticklabels(frase)
        ejes[r, c].set_title(f"Frase {nombre} · {'con PE' if usar_pe else 'sin PE'}")
fig.tight_layout()
plt.show()

# 7 · Visualización 2 · Los embeddings de salida en 2D
Proyecto con PCA (SVD) las salidas de A y B juntas. Cada palabra de A es un círculo y de B una cruz. Sin PE, el círculo y la cruz de cada palabra se superponen: el modelo ve la misma "bolsa de palabras". Con PE se separan, sobre todo "gato" y "perro", que cambian de posición.

In [ ]:
def proyectar(salidas):
    todo = torch.cat(salidas)
    centrado = todo - todo.mean(0)
    _, _, vt = torch.linalg.svd(centrado, full_matrices=False)
    return [(s - todo.mean(0)) @ vt[:2].T for s in salidas]


fig, ejes = plt.subplots(1, 2, figsize=(12, 5))
colores = {"El": "tab:gray", "gato": "tab:orange", "persiguió": "tab:green", "al": "tab:purple", "perro": "tab:blue"}
for ax, usar_pe in zip(ejes, (False, True)):
    sa, _ = ejecutar(FRASE_A, emb, capas, usar_pe)
    sb, _ = ejecutar(FRASE_B, emb, capas, usar_pe)
    pa, pb = proyectar([sa, sb])
    for i, w in enumerate(FRASE_A):
        ax.scatter(*pa[i], c=colores[w], marker="o", s=110)
        ax.annotate(f"{w} (A)", pa[i].tolist(), textcoords="offset points", xytext=(6, 6))
    for i, w in enumerate(FRASE_B):
        ax.scatter(*pb[i], c=colores[w], marker="x", s=110)
    ax.set_title("Con PE" if usar_pe else "Sin PE (círculo = A, cruz = B)")
    ax.grid(alpha=.3)
fig.tight_layout()
plt.show()

# 8 · Visualización 3 · Cuánto cambia cada palabra
Barras con la distancia entre el vector de salida de cada palabra en A y en B (comparando la misma palabra, no la misma posición). Sin PE debería ser cero para todas; con PE, la diferencia es mayor en las palabras que cambiaron de lugar.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, usar_pe in zip(ejes, (False, True)):
    sa, _ = ejecutar(FRASE_A, emb, capas, usar_pe)
    sb, _ = ejecutar(FRASE_B, emb, capas, usar_pe)
    dist = [(sa[PERM[j]] - sb[j]).norm().item() for j in range(len(FRASE_B))]
    ax.bar(FRASE_B, dist, color="tab:red" if usar_pe else "tab:blue")
    ax.set_title("Con PE" if usar_pe else "Sin PE")
    ax.set_ylabel("‖salida(A) − salida(B)‖ por palabra")
fig.tight_layout()
plt.show()

# 9 · Panel interactivo (sliders)
Muevo la dimensión, el número de capas, la escala del embedding, la escala del PE y activo o desactivo el PE. La escala del embedding frente al PE importa: si el embedding es muy grande, el PE queda "tapado" y el modelo casi no nota el orden. En la barra de arriba aparece el error de equivarianza de la configuración elegida.

In [ ]:
def panel(d, capas_n, escala_emb, escala_pe, usar_pe):
    e = crear_embeddings(FRASE_A, d, escala=escala_emb)
    c = crear_pesos(d, capas_n, SEMILLA)
    m = medir(FRASE_A, FRASE_B, e, c, usar_pe, escala_pe)
    _, atA = ejecutar(FRASE_A, e, c, usar_pe, escala_pe)
    _, atB = ejecutar(FRASE_B, e, c, usar_pe, escala_pe)
    fig, ejes = plt.subplots(1, 2, figsize=(10, 4.2))
    for ax, att, fr, n in zip(ejes, (atA[0], atB[0]), (FRASE_A, FRASE_B), "AB"):
        ax.imshow(att.numpy(), cmap="viridis")
        ax.set_xticks(range(5)); ax.set_yticks(range(5))
        ax.set_xticklabels(fr, rotation=45); ax.set_yticklabels(fr)
        ax.set_title(f"Atención capa 1 · frase {n}")
    fig.suptitle(f"{'CON' if usar_pe else 'SIN'} PE · error de equivarianza = {m['error_equivarianza']:.2e} · "
                 f"dif. de la frase = {m['dif_promedio_frase']:.2e}")
    fig.tight_layout()
    plt.show()


w_d = widgets.Dropdown(options=[8, 16, 32, 64], value=16, description="d")
w_c = widgets.IntSlider(value=2, min=1, max=6, description="capas")
w_e = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description="escala emb")
w_p = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description="escala PE")
w_u = widgets.Checkbox(value=True, description="usar PE")
salida = widgets.interactive_output(panel, {"d": w_d, "capas_n": w_c, "escala_emb": w_e, "escala_pe": w_p, "usar_pe": w_u})
display(widgets.VBox([widgets.HBox([w_d, w_c, w_u]), widgets.HBox([w_e, w_p]), salida]))

# 10 · Experimento 2 · Barrido de configuraciones
Repito la comparación con 4 dimensiones × 4 profundidades × 4 escalas de embedding (64 configuraciones), con y sin PE. Sin PE el error debe ser de orden 1e-15 en todas. Con PE depende de cuánto pesa el PE frente al embedding.

In [ ]:
filas = []
for d, nc, esc in itertools.product([8, 16, 32, 64], [1, 2, 4, 6], [0.1, 1.0, 3.0, 10.0]):
    e = crear_embeddings(FRASE_A, d, escala=esc)
    c = crear_pesos(d, nc, SEMILLA)
    for usar_pe in (False, True):
        filas.append({"d": d, "capas": nc, "escala_emb": esc, "usar_pe": usar_pe, **medir(FRASE_A, FRASE_B, e, c, usar_pe)})
e2 = pd.DataFrame(filas)
display(e2.groupby("usar_pe").error_equivarianza.agg(["mean", "min", "max"]))

## 10 · b · Mapa de calor: escala del embedding frente a escala del PE
Con 2 capas y d = 16, el error de equivarianza (con PE) para distintas escalas. La fila `escala PE = 0` es el caso sin PE.

In [ ]:
escalas_emb = [0.1, 0.5, 1.0, 3.0, 10.0]
escalas_pe = [0.0, 0.1, 0.5, 1.0, 3.0]
mapa = np.zeros((len(escalas_pe), len(escalas_emb)))
for i, sp in enumerate(escalas_pe):
    for j, se in enumerate(escalas_emb):
        e = crear_embeddings(FRASE_A, 16, escala=se)
        mapa[i, j] = medir(FRASE_A, FRASE_B, e, crear_pesos(16, 2, SEMILLA), sp > 0, sp)["error_equivarianza"]
plt.figure(figsize=(6.5, 4.5))
plt.imshow(mapa, cmap="magma", aspect="auto")
plt.colorbar(label="error de equivarianza")
plt.xticks(range(len(escalas_emb)), escalas_emb); plt.yticks(range(len(escalas_pe)), escalas_pe)
plt.xlabel("escala del embedding"); plt.ylabel("escala del PE")
for i in range(len(escalas_pe)):
    for j in range(len(escalas_emb)):
        plt.text(j, i, f"{mapa[i, j]:.2g}", ha="center", va="center", color="w", fontsize=8)
plt.tight_layout()
plt.show()

# 11 · Experimento 3 · Más frases y 100 permutaciones aleatorias
Para no depender de un solo par de frases, pruebo la consigna, una frase de 7 palabras y una con la palabra "gato" repetida. En cada una aplico 100 permutaciones aleatorias de las palabras y mido el error de equivarianza y la diferencia de la frase.

In [ ]:
def experimento_permutaciones(frases, n_perm=100, semilla_perm=7):
    rng = np.random.default_rng(semilla_perm)
    filas = []
    for nombre, fr in frases.items():
        e = crear_embeddings(fr, D)
        for usar_pe in (False, True):
            errs, difs = [], []
            for _ in range(n_perm):
                perm = list(rng.permutation(len(fr)))
                sal, _ = ejecutar(fr, e, capas, usar_pe)
                sal_p, _ = ejecutar([fr[j] for j in perm], e, capas, usar_pe)
                errs.append((sal_p - sal[perm]).abs().max().item())
                difs.append((sal.mean(0) - sal_p.mean(0)).abs().max().item())
            filas.append({"frase": nombre, "n_tokens": len(fr), "usar_pe": usar_pe,
                          "err_eq_max": max(errs), "err_eq_media": float(np.mean(errs)),
                          "dif_frase_max": max(difs), "dif_frase_media": float(np.mean(difs))})
    return pd.DataFrame(filas)

In [ ]:
e3 = experimento_permutaciones({"consigna": FRASE_A, **FRASES_EXTRA})
display(e3)

# 12 · Comparación con la referencia en NumPy
Antes de este notebook ejecuté el mismo experimento con una implementación independiente en NumPy (`codigo/referencia_numpy.py`). Aquí comparo las cifras de PyTorch con las que esa referencia guardó. Si coinciden, la implementación en PyTorch es correcta y el resultado no depende de la librería.

In [ ]:
REF_E1_CON_PE = {"error_equivarianza": 1.4740524812860825, "dif_promedio_frase": 0.3611362015415375, "cos_gato_A_vs_B": 0.8685718234278538}
REF_E2 = {False: (1.680947048221526e-15, 1.0658141036401504e-14), True: (0.8010439362690127, 2.283213188372973)}
REF_E3_MEDIA_CON_PE = {"consigna": 1.1970777865842388, "larga_7": 1.3096775059216368, "repetida": 1.287684857867277}

for clave, valor in REF_E1_CON_PE.items():
    assert abs(con_pe[clave] - valor) < 1e-9, clave
for usar_pe, (media, maximo) in REF_E2.items():
    g = e2[e2.usar_pe == usar_pe].error_equivarianza
    if usar_pe:
        assert abs(g.mean() - media) < 1e-9 and abs(g.max() - maximo) < 1e-9
    else:
        assert g.max() < 1e-10 and maximo < 1e-10
for frase, valor in REF_E3_MEDIA_CON_PE.items():
    fila = e3[(e3.frase == frase) & e3.usar_pe].iloc[0]
    assert abs(fila.err_eq_media - valor) < 1e-9, frase
print("PyTorch coincide con la referencia NumPy (con PE a 1e-9; sin PE ambos del orden de 1e-15)")

# 13 · Chequeos de hipótesis
Cada afirmación del análisis, con su valor de verdad calculado.

In [ ]:
chequeos = pd.DataFrame([
    ("Sin PE, permutar la frase solo permuta la salida (error < 1e-10) en las 3 frases y 100 permutaciones", bool((e3[~e3.usar_pe].err_eq_max < 1e-10).all())),
    ("Sin PE, la media de las filas no distingue A de B (dif < 1e-10) en las 3 frases", bool((e3[~e3.usar_pe].dif_frase_max < 1e-10).all())),
    ("Con PE, el error de equivarianza medio supera 0.1 en las 3 frases", bool((e3[e3.usar_pe].err_eq_media > 0.1).all())),
    ("En las 64 configuraciones del barrido, sin PE el error es < 1e-10", bool((e2[~e2.usar_pe].error_equivarianza < 1e-10).all())),
    ("En las 64 configuraciones, con PE el error nunca llega a cero (mínimo > 0.01)", bool(e2[e2.usar_pe].error_equivarianza.min() > 0.01)),
    ("Con embedding de escala 10 el PE pesa menos: el error con PE es menor que con escala 1 (media de las 16 combinaciones de d y capas)",
     bool(e2[e2.usar_pe & (e2.escala_emb == 10.0)].error_equivarianza.mean() < e2[e2.usar_pe & (e2.escala_emb == 1.0)].error_equivarianza.mean())),
], columns=["afirmación", "verdadero"])
pd.set_option("display.max_colwidth", 200)
display(chequeos)

# 14 · Fuentes y qué respaldan
Todas están indexadas en IEEE Xplore o Scopus y son de 2022 a 2026. Leí el resumen completo de cada una (no el texto completo; los PDF los descargo con el script de `codigo/`); la lista con DOI está en `fuentes/fuentes_doi.csv`.

| Afirmación del análisis | Fuente |
| --- | --- |
| Los Transformers no distinguen el orden por sí solos y el PE es lo que lo permite; en configuración no causal el PE ayuda de forma significativa y en una causal "no es muy útil" porque la atención causal puede incorporar la posición de forma implícita | Zhang et al. (2024), ICASSP, DOI 10.1109/ICASSP48485.2024.10446337 (resumen) |
| La self-attention no tiene noción nativa del orden de los tokens (y ese trabajo incluye análisis con PCA y una ablación que quita el PE) | Banđur, Banđur y Jakšić (2026), IEEE TPAMI, DOI 10.1109/TPAMI.2026.3736257 (resumen) |
| En NLP se compararon PE sinusoidal, RoPE, ALiBi y sin PE (NoPos): en análisis de sentimiento RoPE y NoPos rindieron parecido, mientras que ALiBi fue mejor en generación de texto; la necesidad del PE depende de la tarea | Guo (2024), ICEIB, DOI 10.1109/ICEIB61477.2024.10602658 (resumen) |
| Un modelo sin PE con máscara causal sí puede inferir posición (límite de mi experimento) | Hayakawa y Sato (2025), ACL, DOI 10.18653/v1/2025.acl-long.1488 |
| El PE como eje central para extrapolar a secuencias largas | Zhao et al. (2024), EMNLP Findings, DOI 10.18653/v1/2024.findings-emnlp.582 |
| Add & Norm (LayerNorm tras la suma residual) es lo que usa cada capa del codificador, y modificarlo condiciona la estabilidad de redes profundas | Wang et al. (2024), IEEE TPAMI, DOI 10.1109/TPAMI.2024.3386927 |

# 15 · Interpretación y límites
**Lo que muestra el experimento.** Sin PE, la salida del codificador para cada palabra no depende de su posición: el error de equivarianza es del orden de 1e-15 en las 3 frases, las 100 permutaciones y las 64 configuraciones del barrido. Por eso "El gato persiguió al perro" y "El perro persiguió al gato" no se pueden distinguir con la media de los vectores, y el "gato" tiene el mismo vector en ambas (coseno 1). Con PE las salidas cambian y la diferencia depende de cuánto pesa el PE frente al embedding.

**Por qué ocurre.** Los pesos `Wq`, `Wk`, `Wv` son los mismos para todas las posiciones y el softmax suma sobre todas las posiciones por igual. Permutar la entrada solo permuta las filas y columnas de la matriz de atención. El orden solo entra al modelo si algo lo escribe en el vector de cada token: eso es el PE.

**Límites.**
- Uso pesos aleatorios y un codificador bidireccional sin entrenar. Muestra una propiedad estructural; no mide cuánto empeoraría una tarea real.
- Con una máscara causal (decoder) el modelo **sí puede** inferir la posición sin PE (Hayakawa y Sato, 2025; Zhang et al., 2024, observan que en configuración causal el PE no ayuda mucho). Mi resultado vale para el codificador bidireccional de la Clase 12, no para todo Transformer sin PE.
- La conclusión de cuánto "se pierde" con PE depende de la escala del embedding (sección 10).
- Que el orden se pierda no significa que toda tarea empeore: en análisis de sentimiento NoPos rindió parecido a RoPE (Guo, 2024). El orden importa en tareas como la de la consigna, donde quién persigue a quién cambia el significado.
- Un solo tipo de PE (sinusoidal); no probé aprendido, RoPE ni ALiBi.

# 16 · Guardar resultados
Guardo las tablas y los chequeos como CSV y los comprimo para descargarlos. Si no estás en Colab, solo se guardan en la carpeta local.

In [ ]:
e1.to_csv(CARPETA_SALIDA / "e1_consigna_torch.csv", index=False)
e2.to_csv(CARPETA_SALIDA / "e2_barrido_torch.csv", index=False)
e3.to_csv(CARPETA_SALIDA / "e3_permutaciones_torch.csv", index=False)
chequeos.to_csv(CARPETA_SALIDA / "chequeos_hipotesis_torch.csv", index=False)
print(sorted(p.name for p in CARPETA_SALIDA.iterdir()))

In [ ]:
import shutil
shutil.make_archive("resultados_s13", "zip", CARPETA_SALIDA)
try:
    from google.colab import files
    files.download("resultados_s13.zip")
except ImportError:
    print("Fuera de Colab: el zip quedó en el directorio actual")